# Landsat 8/9 Processing Workflow

This notebook demonstrates the complete workflow for processing Landsat 8/9 satellite imagery using the `disasters-product-algorithms` package.

## Workflow Steps
1. **Download Landsat Data** - Download from USGS EarthExplorer (external)
2. **Configure Environment Variables** - Set processing parameters
3. **Process Landsat Data** - Generate products with COG conversion and event naming
4. **View Results** - Examine the generated outputs

## Features Demonstrated
- Cloud Optimized GeoTIFF (COG) conversion with customizable compression
- Event-based file naming for disaster response
- Multiple product generation (true color, panchromatic, natural color, NDVI, NDWI, MNDWI, EVI, NBR, water extent)
- Cloud masking
- Automatic merging by date and product

## 1. Download Landsat Data (External)

**Before running this notebook, download Landsat 8/9 data from:**
- [USGS EarthExplorer](https://earthexplorer.usgs.gov/)
- [NASA Earthdata](https://earthdata.nasa.gov/)

**Requirements:**
- Landsat Collection 2 Level-2 surface reflectance data
- Format: `.tar` or `.zip` files
- Place all `.tar` or `.zip` files in a single directory

## 2. Environment Setup

Configure all processing parameters as environment variables for easy modification.

In [ ]:
import os
import subprocess
from pathlib import Path

# ==================== CONFIGURATION ====================
# Modify these variables according to your requirements

# Input directory containing Landsat .tar or .zip files
INPUT_DIR = os.path.expanduser("~/disasters_data/landsat")

# Processing parameters
PRODUCTS = ["true", "pan", "nat", "ndvi", "ndwi", "mndwi", "evi", "nbr"]  # Products to generate
EVENT_NAME = "202311_Example_Event"                                        # Event prefix for output files
PROCESS_DATE = None                                                        # Specific date to process (YYYYMMDD format, None = all dates)
PROCESS_TILE = None                                                        # Specific tile to process (path/row, e.g., "171035", None = all tiles)

# Water extent parameters (if "we" in PRODUCTS)
WATER_EXTENT_NSTD = [1, 1.5]   # Standard deviations for water extent classification

# COG options
COMPRESSION = "ZSTD"            # Compression type: ZSTD, DEFLATE, LZW
COMPRESSION_LEVEL = 22          # Compression level (22 for ZSTD, 9 for DEFLATE)
NODATA = None                   # No-data value (None = auto-detect)
TIF_ONLY = False                # Set to True to skip COG conversion

# Processing flags
ENABLE_MERGE = True             # Merge tiles by date and product
ENABLE_MASK = True              # Apply cloud masking
FORCE_OVERWRITE = False         # Force reprocessing of existing files
UNZIP_ONLY = False              # Only unzip files, don't process

# Create input directory if it doesn't exist
os.makedirs(INPUT_DIR, exist_ok=True)

# Target CRS for the output COG.
# None preserves the source projection (fastest, no warp).
# Uncomment the EPSG:3857 line if you plan to push the COG through
# veda-data-airflow's build_stac, which trips on the EPSG:4326 ensemble.
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"

print("Configuration:")
print(f"  Input Directory: {INPUT_DIR}")
print(f"  Products: {', '.join(PRODUCTS)}")
print(f"  Event Name: {EVENT_NAME}")
print(f"  Process Date: {PROCESS_DATE if PROCESS_DATE else 'All dates'}")
print(f"  Process Tile: {PROCESS_TILE if PROCESS_TILE else 'All tiles'}")
print(f"  COG Compression: {COMPRESSION} (level {COMPRESSION_LEVEL})")
print(f"  Merge: {ENABLE_MERGE}")
print(f"  Cloud Masking: {ENABLE_MASK}")

## 3. Verify Input Data

Check that Landsat files are present in the input directory.

In [ ]:
import glob

# Check for .tar and .zip files
tar_files = glob.glob(os.path.join(INPUT_DIR, "*.tar"))
zip_files = glob.glob(os.path.join(INPUT_DIR, "*.zip"))
all_files = tar_files + zip_files

print(f"Found {len(all_files)} Landsat file(s) in {INPUT_DIR}:\n")

if all_files:
    for f in all_files:
        file_size = os.path.getsize(f) / (1024*1024*1024)  # Size in GB
        print(f"  - {os.path.basename(f)} ({file_size:.2f} GB)")
    print("\n✓ Ready to process!")
else:
    print("⚠ No Landsat files found!")
    print("\nPlease download Landsat data from USGS EarthExplorer and place in:")
    print(f"  {INPUT_DIR}")

## 4. Process Landsat Data

Process the Landsat imagery to generate various products with COG conversion and event naming.

**Note:** The processing script has been configured with unbuffered output to display progress in real-time within JupyterHub. You'll see:
- Progress bars for scene processing
- Detailed product generation steps
- COG conversion progress
- Error messages if any products fail
- Final processing summary with success/failure counts
- Log file location for detailed error tracking

In [ ]:
# Build processing command
process_cmd = [
    "process_landsat89",
    INPUT_DIR,
    "-p", *PRODUCTS,
    "-event", EVENT_NAME,
    "-compression", COMPRESSION,
    "-compression_level", str(COMPRESSION_LEVEL),
    "-dst_crs", TARGET_CRS if TARGET_CRS else "native",
]

# Add optional parameters
if PROCESS_DATE:
    process_cmd.extend(["-date", PROCESS_DATE])

if PROCESS_TILE:
    process_cmd.extend(["-tile", PROCESS_TILE])

if "we" in PRODUCTS:
    process_cmd.extend(["-we_nstd", *[str(n) for n in WATER_EXTENT_NSTD]])

if ENABLE_MERGE:
    process_cmd.append("-merge")

if ENABLE_MASK:
    process_cmd.append("-mask")

if FORCE_OVERWRITE:
    process_cmd.append("-force")

if UNZIP_ONLY:
    process_cmd.append("-unzip_only")

if TIF_ONLY:
    process_cmd.append("-tif_only")

if NODATA is not None:
    process_cmd.extend(["-nodata", str(NODATA)])

print("Processing Landsat data...")
print(f"Command: {' '.join(process_cmd)}")
print()

# Execute processing with real-time output
# The scripts have built-in unbuffered output for JupyterHub compatibility
process = subprocess.Popen(
    process_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1  # Line buffered
)

# Print output in real-time
for line in process.stdout:
    print(line, end='')  # end='' because line already has newline

# Wait for completion
return_code = process.wait()

if return_code == 0:
    print("\n✓ Processing completed successfully!")
else:
    print(f"\n✗ Processing failed with return code {return_code}")

## 5. View Results

Examine the generated output files and directory structure.

In [ ]:
import glob

# Find output directory
output_dir = os.path.join(INPUT_DIR, "output")

if os.path.exists(output_dir):
    print(f"Output directory: {output_dir}\n")
    
    # List all generated products
    print("Generated Products:")
    print("=" * 80)
    
    # Find all GeoTIFF files
    tif_files = sorted(glob.glob(os.path.join(output_dir, "**/*.tif"), recursive=True))
    
    if tif_files:
        # Group by product type
        product_types = {}
        for tif_file in tif_files:
            # Extract product type from path
            parts = tif_file.split(os.sep)
            if len(parts) >= 3:
                product_type = parts[-2]  # Directory name (e.g., trueColor, NDVI)
                if product_type not in product_types:
                    product_types[product_type] = []
                product_types[product_type].append(os.path.basename(tif_file))
        
        # Display grouped results
        for product_type, files in sorted(product_types.items()):
            print(f"\n{product_type}:")
            for f in files:
                file_size = os.path.getsize(os.path.join(output_dir, product_type, f)) / (1024*1024)
                print(f"  - {f} ({file_size:.1f} MB)")
        
        print(f"\nTotal files: {len(tif_files)}")
    else:
        print("No GeoTIFF files found.")
else:
    print(f"Output directory not found: {output_dir}")

## Output File Naming Convention

Files are named with the event prefix and formatted date:

**Original:** `LC08_trueColor_20250922_185617_046028.tif`

**With Event Naming:** `202311_Example_Event_LC08_trueColor_185617_046028_2025-09-22_day.tif`

This naming convention:
- Adds event prefix for organization
- Removes date from middle position
- Adds formatted date (YYYY-MM-DD) at the end
- Includes `_day` suffix for AWS/cloud compatibility

## Product Descriptions

| Product | Description | Use Case |
|---------|-------------|----------|
| **true** | True color (RGB) | Visual interpretation |
| **pan** | Panchromatic (Band 8) | High-resolution grayscale |
| **nat** | Natural color | Natural-looking imagery |
| **colorIR** | Color infrared | Vegetation health |
| **ndvi** | Normalized Difference Vegetation Index | Vegetation density |
| **ndwi** | Normalized Difference Water Index | Water detection |
| **mndwi** | Modified NDWI | Enhanced water detection |
| **evi** | Enhanced Vegetation Index | Improved vegetation monitoring |
| **nbr** | Normalized Burn Ratio | Burn severity assessment |
| **we** | Water extent classification | Flood mapping |

## Next Steps

You can now:
1. Load and visualize the GeoTIFF files using libraries like `rasterio` or `GDAL`
2. Upload the COG files to cloud storage (S3, GCS, etc.)
3. Process additional dates or tiles by modifying the configuration variables
4. Generate additional products by updating the `PRODUCTS` list
5. Experiment with different compression settings for optimal file sizes